# Análise completa do conjunto de dados e revisão do trabalho de melhoria

Este notebook conduz duas tarefas. A primeira é a revisão crítica do trabalho de melhoria do conjunto de dados já realizado, tratando cada resultado anterior como afirmação a verificar e não como fato. A segunda é a análise do conjunto feita do zero, cobrindo qualidade, estrutura, cobertura do espaço de configurações e o que o texto livre da base ainda pode render.

As possibilidades de melhoria, testadas e não testadas, são consolidadas na Parte 3 e no relatório que acompanha este notebook.

Todo número reportado obedece ao protocolo do projeto: validação cruzada deixando uma configuração de fora por vez, transformações e escolhas refeitas dentro de cada dobra, comparações pareadas por leitura, e conjunto de teste reservado intocado.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S3/analise_dataset"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import io
import json
import re
from collections import Counter, defaultdict
from itertools import combinations

import numpy as np
import pandas as pd

BASE = "../../colabs_S4/"
# A versao de perfil de perdas nao foi levada para a quarta sprint; ela e lida
# da pasta da terceira apenas onde a comparacao entre as duas e o proprio objeto.
BASE_PERFIL = "../"
PARTICAO = "../preparacao_testes/particao_treino_teste.json"
BRUTO = "../../../documents/outros/dados_whirlpool.csv"
PREFIXOS = ("blower_direction_", "housing_", "board_position_", "run_case_", "losses_",
            "perfil_perdas_", "deflector_hole_", "material_coil_support_", "cfg_",
          "reason_", "loss_profile_order_")
AREAS = ["coil_temps", "ct_knobs", "ipc_comp", "ipc_cooling", "main_vent_sys",
         "mainback", "oven_hmi"]
PISO_AFIRMADO = {"oven_hmi": 0.58, "ct_knobs": 0.92, "main_vent_sys": 1.16, "ipc_cooling": 2.15,
                 "mainback": 1.33, "ipc_comp": 3.30, "coil_temps": 6.36}

particao = json.load(open(PARTICAO, encoding="utf-8"))
DOBRA = {int(k): v for k, v in particao["dobra_por_run_idx"].items()}
TREINO, TESTE = set(particao["run_idx_treino"]), set(particao["run_idx_teste"])
replicata = json.load(open(BASE + "chave_replicata.json", encoding="utf-8"))
CHAVE = {int(k): v for k, v in replicata["chave_por_run_idx"].items()}

# A versao adotada pelo projeto e a de treze categorias. A de perfil de perdas
# permanece carregada porque algumas secoes comparam explicitamente as duas;
# fora dessas, a base de trabalho e sempre a adotada.
dados = {a: {"13cat": pd.read_csv(BASE + f"dataset_{a}_13cat.csv"),
             "perfil": pd.read_csv(BASE_PERFIL + f"dataset_{a}_perfil.csv")} for a in AREAS}


def alvos_de(df):
    return [c for c in df.columns if not c.startswith(PREFIXOS)]


def entradas_de(df):
    return [c for c in df.columns if c.startswith(PREFIXOS)]


print("dados carregados para", len(dados), "regioes")

## Parte 1. Revisão crítica do trabalho anterior

### 1.1. Estrutura dos dados, afirmações contra os arquivos

Cada afirmação recebida sobre a estrutura é conferida diretamente nos arquivos. Duas delas revelaram-se imprecisas, conforme registrado nas saídas.

In [ ]:
for a in AREAS:
    db, dperfil = dados[a]["13cat"], dados[a]["perfil"]
    ent = entradas_de(db)
    binarias = sum(1 for c in ent if set(db[c].dropna().unique()) <= {0, 1})
    print(f"  {a:<14} linhas {len(db):>3} | entradas {len(ent):>2} (binarias {binarias:>2}) "
          f"| alvos {len(alvos_de(db)):>2} | versao perfil: {len(entradas_de(dperfil))} entradas")

db0 = dados["ipc_comp"]["13cat"]
ent0 = entradas_de(db0)
print(f"\n  configuracoes distintas no espaco de entrada: {len(db0[ent0].drop_duplicates())}  [afirmado: 41]")
print("  CORRECAO 1: nao existe coluna run_idx nos arquivos; o indice do ensaio e a posicao da linha.")

print("\n  diferenca entre as versoes _perfil e _13cat, alem das colunas de perda:")
for a in AREAS:
    db, dperfil = dados[a]["13cat"], dados[a]["perfil"]
    comuns = sorted((set(db.columns) & set(dperfil.columns)))
    difere = [c for c in comuns if not np.allclose(db[c].fillna(-9e9), dperfil[c].fillna(-9e9))]
    so_p = sorted(set(dperfil.columns) - set(db.columns) - {c for c in dperfil.columns if c.startswith("losses_")})
    so_13 = sorted(set(db.columns) - set(dperfil.columns) - {c for c in db.columns if c.startswith("losses_")})
    print(f"    {a:<14} colunas comuns com valores diferentes: {difere} | so numa versao (fora perdas): {so_p + so_13 or 'nenhuma'}")
print("  CORRECAO 2: as duas versoes possuem colunas losses_miro e losses_simone com o MESMO nome e")
print("  valores DIFERENTES: na versao 13cat sao os rotulos literais (2 e 1 ensaios), na versao perfil")
print("  sao as familias (10 e 32 ensaios). Uma juncao por nome de coluna entre as versoes seria incorreta.")

### 1.2. Partição e dobras

A afirmação recebida era de 17 dobras, com os ensaios de teste sem dobra atribuída. Uma primeira leitura do arquivo contou 18 valores distintos, o que teria indicado vazamento dos ensaios de teste para a validação. A leitura estava errada: o décimo oitavo valor é nulo, e corresponde exatamente aos dez ensaios reservados.

In [ ]:
numericas = {k: v for k, v in DOBRA.items() if v is not None}
nulas = sorted(k for k, v in DOBRA.items() if v is None)
print(f"  ensaios de treino {len(TREINO)} | teste {len(TESTE)} | intersecao {len(TREINO & TESTE)}")
print(f"  dobras numericas: {len(set(numericas.values()))}  [afirmado: 17]")
print(f"  ensaios com dobra nula: {nulas}")
print(f"  coincidem com os ensaios de teste: {set(nulas) == TESTE}")
tam = Counter(numericas.values())
print(f"  ensaios por dobra: min {min(tam.values())} | mediana {int(np.median(list(tam.values())))} | max {max(tam.values())}")
print("  VERIFICADO: nao ha vazamento do teste para a validacao. A dobra nula e o mecanismo que exclui o teste.")

### 1.3. Piso de ruído, reproduzido a partir da chave de replicata

O piso de cada região é a raiz da variância agrupada dentro dos grupos de repetição, ponderada por graus de liberdade. Os sete valores afirmados são reproduzidos exatamente. A seção 2.5 mostra, porém, que um dos seis pares não é replicata de fato, e que o piso de uma região está inflado por ele.

In [ ]:
grupos = defaultdict(list)
for r, ch in CHAVE.items():
    grupos[ch].append(r)
PARES = {ch: rs for ch, rs in grupos.items() if len(rs) > 1}
print(f"  configuracoes na chave: {replicata['n_configuracoes']} | grupos de repeticao: {len(PARES)} | "
      f"tamanhos {sorted(len(v) for v in PARES.values())} | ensaios cobertos {sum(len(v) for v in PARES.values())}")
print(f"  grupos inteiramente no treino: {sum(1 for v in PARES.values() if all(r in TREINO for r in v))} de {len(PARES)}")


def piso_de(area, grupos_usados):
    db = dados[area]["13cat"]
    ss, gl = 0.0, 0
    for rs in grupos_usados:
        for c in alvos_de(db):
            v = db.loc[rs, c].dropna().to_numpy()
            if len(v) > 1:
                ss += ((v - v.mean()) ** 2).sum()
                gl += len(v) - 1
    return (float(np.sqrt(ss / gl)) if gl else float("nan")), gl


print(f"\n  {'regiao':<14}{'afirmado':>10}{'reproduzido':>13}{'gl':>5}{'bate':>6}")
for a in AREAS:
    p, gl = piso_de(a, PARES.values())
    print(f"  {a:<14}{PISO_AFIRMADO[a]:>10.2f}{p:>13.2f}{gl:>5}{'sim' if abs(p - PISO_AFIRMADO[a]) < .05 else 'NAO':>6}")

## Parte 2. Análise do conjunto de dados

### 2.1. Ausentes

In [ ]:
for a in AREAS:
    db = dados[a]["13cat"]
    alv = alvos_de(db)
    na_c = db[alv].isna().sum()
    na_r = db[alv].isna().sum(axis=1)
    top = ", ".join(f"{c[:26]} ({n})" for c, n in na_c[na_c > 0].sort_values(ascending=False).head(3).items())
    print(f"  {a:<14} ausentes {int(na_c.sum()):>3} de {len(db) * len(alv):>5} ({100 * na_c.sum() / (len(db) * len(alv)):4.1f}%) "
          f"| componentes com ausente {int((na_c > 0).sum()):>2} de {len(alv):>2} | ensaios completos {int((na_r == 0).sum()):>2}"
          + (f"\n  {'':<14}   mais ausentes: {top}" if top else ""))
print("\n  Observacao: ipc_comp possui apenas 22 ensaios completos nos 37 componentes, o que restringe")
print("  qualquer analise que exija a matriz inteira, como a decomposicao em valores singulares.")

### 2.2. A coluna de perdas: os treze rótulos e a decomposição inferida

A decomposição do rótulo em família, ordem e indicador de perfil modificado foi inferida do texto. O mapa completo é exibido para que a inferência possa ser conferida rótulo a rótulo. A leitura do texto bruto, na seção 2.4, corrige o significado atribuído ao termo ordem.

In [ ]:
d13 = dados["ipc_comp"]["13cat"]
cols13 = [c for c in d13.columns if c.startswith("losses_")]
rotulos = []
for i in range(len(d13)):
    marcados = [c[len("losses_"):] for c in cols13 if d13.at[i, c] == 1]
    rotulos.append(marcados[0] if len(marcados) == 1 else "")
rotulos = pd.Series(rotulos)
n_um = int(sum(1 for i in range(len(d13)) if int(d13.loc[i, cols13].sum()) == 1))
print(f"  ensaios com exatamente um rotulo: {n_um} | sem rotulo: {int((rotulos == '').sum())} | com mais de um: {56 - n_um - int((rotulos == '').sum())}")
SEM_ROTULO = sorted(d13.index[rotulos == ""].tolist())
print(f"  ensaios sem rotulo (posicao): {SEM_ROTULO}  | destes, no teste: {sorted(set(SEM_ROTULO) & TESTE)}")


def familia(r):
    if r == "": return "sem rotulo"
    if "no losses" in r: return "no losses"
    if "simone" in r: return "simone"
    if "miro" in r: return "miro"
    return "outro"


def ordem(r):
    if r == "" or "no losses" in r: return "nao se aplica"
    if "front order" in r: return "front"
    if "back order" in r: return "back"
    return "nao declarada"


print(f"\n  {'n':>3}  {'rotulo':<40} {'familia':<11} {'ordem':<14} mod  num")
for r in sorted(set(rotulos)):
    n = int((rotulos == r).sum())
    print(f"  {n:>3}  {(r or '(vazio)'):<40} {familia(r):<11} {ordem(r):<14} {int('modified' in r)}    {int(bool(re.search(r'\d{2,4}/\d{2,4}', r)))}")
sup = Counter((familia(r), ordem(r)) for r in rotulos)
print("\n  suporte minimo por nivel: 13 categorias -> 1 ensaio | familia -> "
      f"{min(Counter(map(familia, rotulos)).values())} | ordem -> {min(Counter(map(ordem, rotulos)).values())}")

### 2.3. Componentes e o limite

Oito dos 77 pontos são de monitoramento, sem limite rígido; eles têm limite nulo na tabela e ficam fora de qualquer contagem de decisão. Entre os 69 restantes, 34 atingem o limite em pelo menos um ensaio, concentrados em três regiões.

In [ ]:
limites = pd.read_csv(BASE + "limites_ativos.csv")
LIM = {str(r["tc_name"]).replace("\n", ", ").strip(): r["limite_num"]
       for _, r in limites.iterrows() if pd.notna(r["limite_num"])}
print(f"  componentes com limite numerico: {len(LIM)} | apenas monitoramento: {int(limites['apenas_monitorar'].sum())}")
print("  monitoramento:", ", ".join(limites.loc[limites["apenas_monitorar"], "tc_name"].str[:26]))
atinge = []
for a in AREAS:
    db = dados[a]["13cat"]
    for c in alvos_de(db):
        if c in LIM and db[c].max() >= LIM[c]:
            atinge.append((a, c, LIM[c] - db[c].max()))
print(f"\n  componentes que atingem o limite em algum ensaio: {len(atinge)} | por regiao: {dict(Counter(a for a, _, _ in atinge))}")
print("  os cinco que mais excedem:")
for a, c, f in sorted(atinge, key=lambda t: t[2])[:5]:
    print(f"    {a:<14} {c:<30} {f:>+7.1f} C")

### 2.4. O texto livre da base bruta

A base original está em formato largo, com um ensaio por coluna. O texto livre encontra-se em dois lugares: na linha Reason, já explorada anteriormente para extrair seis indicadores, e nos cabeçalhos das colunas, que nunca haviam sido lidos. Os cabeçalhos carregam a numeração de um planejamento de experimentos da parceira, marcadores de repetição e descrições de alteração.

A leitura conjunta esclarece quatro pontos que estavam em aberto.

In [ ]:
bruto = pd.read_csv(BRUTO)
cabecalhos = list(bruto.columns[4:])
linha = lambda nome: bruto.loc[bruto["TC Number"] == nome].iloc[0, 4:].tolist()
reason, losses_bruto = linha("Reason"), linha("Losses")
assert len(cabecalhos) == 56

print("  1. Os seis ensaios sem rotulo de perda sao a serie New Baseline com variantes de bandeja:")
for i in SEM_ROTULO:
    print(f"       [{i:>2}] {cabecalhos[i]:<12} | Reason: {reason[i]} | Losses: {losses_bruto[i]}")
print("     E por isso que losses e a alteracao de bandeja tem V de Cramer 0,904: sao os mesmos")
print("     ensaios. Na codificacao atual a bandeja integra o nivel air_path da familia do motivo.")

print("\n  2. O termo order refere-se a ordem dos IGBTs, nao das bobinas:")
for i, (h, r) in enumerate(zip(cabecalhos, reason)):
    if "order" in str(r).lower() or "igbt" in str(r).lower():
        print(f"       [{i:>2}] {h:<12} | Reason: {r} | Losses: {losses_bruto[i]}")
print("     CORRECAO 3: a variavel ordem existe, mas descreve a sequencia de chaveamento dos IGBTs.")

print("\n  3. Repeticoes declaradas nos cabecalhos, e se a chave de replicata as reconhece:")
base_h = [re.sub(r"\.\d+$", "", h).replace(" - Rerunned", "").strip() for h in cabecalhos]
for b, n in Counter(base_h).items():
    if n > 1:
        idx = [i for i, x in enumerate(base_h) if x == b]
        print(f"       {b[:34]:<34} posicoes {idx} | mesma chave: {len({CHAVE[i] for i in idx}) == 1} "
              f"| Reason: {[str(reason[i])[:22] for i in idx]}")

print("\n  4. Run 44, 44.1 e 44.2 tem o mesmo Reason; so o 44.2 traz os numeros no rotulo de perda:")
for i in (39, 41, 43):
    print(f"       [{i}] {cabecalhos[i]:<10} | {reason[i]} | {losses_bruto[i]}")
db = dados["coil_temps"]["13cat"]
alv = alvos_de(db)
desv_442 = (db.loc[43, alv] - db.loc[[39, 41], alv].mean()).abs().mean()
desv_par = (db.loc[39, alv] - db.loc[41, alv]).abs().mean()
print(f"     desvio medio do 44.2 em relacao ao par 44/44.1: {desv_442:.2f} C | desvio entre 44 e 44.1: {desv_par:.2f} C")
print("     O 44.2 fica dentro do espalhamento do par: e a mesma serie, e o rotulo com numeros")
print("     designa o mesmo perfil que Broil C2 Simone - Front Order.")

print("\n  5. Ha tambem informacao numerica solta no texto:")
for i, (h, r) in enumerate(zip(cabecalhos, reason)):
    if re.search(r"\d+ ?(rpm|cfm)", str(r).lower() + " " + h.lower()):
        print(f"       [{i:>2}] {h:<12} | {r}")

### 2.5. O par de replicatas que não é replicata

O par de maior discrepância dentro da chave de replicata é o de posições 16 e 17, com 32,6 graus de diferença numa placa de alumínio. Os cabeçalhos mostram que são os pontos 7 e 8 de um planejamento de experimentos sobre a posição do soprador. A posição variava entre eles, mas as colunas codificadas só registram frente ou trás e aberto ou fechado, de modo que os dois receberam a mesma chave.

O piso de ruído da região das bobinas está, portanto, inflado por um par que não é repetição. O valor corrigido é calculado abaixo, com o par removido e com a tripla 44, 44.1 e 44.2 reconhecida como grupo.

In [ ]:
doe = [i for i, h in enumerate(cabecalhos) if "DoE Run" in h]
dbi = dados["ipc_comp"]["13cat"]
ent = entradas_de(dbi)
print(f"  ensaios do DoE de posicao: {doe} | configuracoes distintas entre eles: {len(dbi.loc[doe, ent].drop_duplicates())}")
print("  cabecalhos das posicoes 15, 16 e 17:", [cabecalhos[i][:44] for i in (15, 16, 17)])
print("  as tres tem codificacao identica:", len(dbi.loc[[15, 16, 17], ent].drop_duplicates()) == 1)

db = dados["coil_temps"]["13cat"]
for c in alvos_de(db):
    x, y = db.at[16, c], db.at[17, c]
    if pd.notna(x) and pd.notna(y) and abs(x - y) > 10:
        print(f"  [16] vs [17] em {c:<22} {x:>7.1f} vs {y:>7.1f}  dif {abs(x - y):>5.1f} C")

sem_par = [v for v in PARES.values() if set(v) != {16, 17}]
com_tripla = [v if set(v) != {39, 41} else [39, 41, 43] for v in sem_par]
PISO_REVISADO = {}
print(f"\n  {'regiao':<14}{'6 pares':>9}{'sem 16/17':>11}{'sem 16/17, tripla 44':>22}")
for a in AREAS:
    p0, _ = piso_de(a, PARES.values())
    p1, _ = piso_de(a, sem_par)
    p2, _ = piso_de(a, com_tripla)
    PISO_REVISADO[a] = p2
    print(f"  {a:<14}{p0:>9.2f}{p1:>11.2f}{p2:>22.2f}")
print("\n  CORRECAO 4: o piso de coil_temps passa de 6,36 para cerca de 5,0 C. A folga redutivel dessa")
print("  regiao e maior do que o afirmado, e ela sobe na ordenacao de onde o esforco rende.")

### 2.6. Cobertura e confundimento do espaço de configurações

In [ ]:
grp = defaultdict(list)
for c in ent0:
    for p in PREFIXOS:
        if c.startswith(p):
            grp[p].append(c)
            break
fat = pd.DataFrame(index=db0.index)
for p, cols in grp.items():
    if p == "cfg_":
        for c in cols:
            fat[c] = db0[c].astype(int)
    else:
        lv = db0[cols].to_numpy()
        fat[p.rstrip("_")] = [cols[np.where(lv[i] == 1)[0][0]][len(p):] if (lv[i] == 1).any() else "ausente"
                              for i in range(len(db0))]
print("  niveis com 5 ou menos ensaios:")
for c in fat.columns:
    for k, v in fat[c].value_counts().items():
        if v <= 5 and not (c.startswith("cfg_") and k == 0):
            print(f"    {c} = {k}: {v}")


def cramer(a, b):
    t = pd.crosstab(a, b).to_numpy()
    n = t.sum()
    if min(t.shape) < 2:
        return np.nan
    e = np.outer(t.sum(1), t.sum(0)) / n
    return float(np.sqrt(np.nansum((t - e) ** 2 / np.where(e == 0, np.nan, e)) / (n * (min(t.shape) - 1))))


pares_v = sorted(((cramer(fat[a], fat[b]), a, b) for a, b in combinations(fat.columns, 2)), reverse=True)
print("\n  pares mais confundidos (V de Cramer):")
for v, a, b in pares_v[:5]:
    print(f"    {v:.3f}  {a} x {b}")
vazias = sum(1 for a, b in combinations(fat.columns, 2)
             for x in fat[a].unique() for y in fat[b].unique()
             if (x, y) not in set(map(tuple, fat[[a, b]].to_numpy())))
print(f"\n  celulas de dois fatores nunca ensaiadas: {vazias}  [afirmado: 84]")

### 2.7. Estrutura entre componentes, por região

A matriz de leituras de cada região, centrada por componente, é decomposta em valores singulares. O número de modos necessário e o resíduo com dois e três modos são comparados ao piso de ruído revisado.

In [ ]:
print(f"  {'regiao':<14}{'ensaios':>8}{'comp':>6}{'modos 95%':>11}{'MAE k=2':>9}{'MAE k=3':>9}{'piso rev.':>10}")
for a in AREAS:
    db = dados[a]["13cat"]
    M = db[alvos_de(db)].dropna().to_numpy(float)
    C = M - M.mean(0)
    U, S, Vt = np.linalg.svd(C, full_matrices=False)
    k95 = int(np.searchsorted(np.cumsum(S ** 2) / (S ** 2).sum(), .95) + 1)
    mae = lambda k: np.abs(C - U[:, :k] @ np.diag(S[:k]) @ Vt[:k]).mean()
    print(f"  {a:<14}{len(M):>8}{M.shape[1]:>6}{k95:>11}{mae(2):>9.2f}{mae(3):>9.2f}{PISO_REVISADO[a]:>10.2f}")

### 2.8. Lista priorizada de ensaios novos para a parceira

A lista decorre da cobertura e do confundimento medidos acima, e não de otimização combinatória: uma seleção gulosa que maximiza células fechadas produz configurações fisicamente implausíveis, com vários indicadores ligados ao mesmo tempo. A lista abaixo é a que pode ser defendida.

In [ ]:
print("  1. Quebrar o confundimento entre perfil de perda e bandeja. Os seis ensaios de bandeja sao os seis")
print("     sem rotulo de perda. Pedir os tres tipos de bandeja com perfil de perda declarado, e confirmar")
print("     se nesses seis ensaios a placa operava e com que perfil.")
print("  2. Completar os niveis fracos com um unico fator alterado sobre a configuracao mais frequente:")
cfg_comum = db0[ent0].value_counts()
base_cfg = dict(zip(ent0, cfg_comum.index[0]))
print(f"     base ({int(cfg_comum.iloc[0])} ensaios): " + ", ".join(k for k, v in base_cfg.items() if v == 1 and not k.startswith(('cfg_', 'reason_'))))
# Os tres ultimos fatores deixaram de ter coluna propria: eles integram o nivel
# air_path da familia do motivo. O pedido a parceira continua valendo, porque ele
# e sobre ensaios a realizar e nao sobre a forma de codifica-los.
for col, rot in [("board_position_right", "placa a direita, hoje 3 ensaios"),
                 ("reason_air_path, variante de defletor", "deflector variante, hoje 3"),
                 ("reason_air_path, vazao elevada", "vazao alta, hoje 5"),
                 ("reason_air_path, bandeja alterada", "bandeja alterada, hoje 5")]:
    print(f"     - {rot:<34} -> base com {col} ligado")
print("  3. Posicao real do soprador nos tres ensaios do DoE de posicao com codificacao identica (15, 16, 17).")
print("  4. Confirmacao de que o rotulo simone corresponde ao perfil 240/145/180/180, e o valor de miro.")
print("     Confirmado, 32 dos 56 ensaios ganham potencia numerica por bobina.")
print("  5. Duas linhas numericas sem nome no fim da base bruta, com valores para os 18 primeiros ensaios,")
print("     nunca identificadas. Se uma for a temperatura ambiente do laboratorio, e a variavel que")
print("     faltou a formulacao em dois estagios.")

## Parte 3. Possibilidades de melhoria, testadas sob o protocolo

### 3.1. Máquina de avaliação

Uma única função avalia qualquer variante sob validação cruzada por configuração. As estatísticas por componente são calculadas dentro de cada dobra. Toda variante altera uma única coisa em relação à referência, para que o efeito seja atribuível.

In [ ]:
import warnings

warnings.filterwarnings("ignore")
from scipy import stats
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing import StandardScaler

PARAMS_RF = dict(n_estimators=300, n_jobs=-1)


def avalia(X, Y, alvos, semente=42, log_alvo=False, excluir=(), media_replicatas=False,
           estimador="rf"):
    """Validacao por configuracao. Devolve {(dobra, componente, ensaio): (previsto, real)}."""
    runs = np.arange(len(Y))
    dob = np.array([DOBRA.get(int(r)) for r in runs], dtype=object)
    if media_replicatas:
        grupos = defaultdict(list)
        for r in runs:
            if dob[r] is not None:
                grupos[CHAVE[int(r)]].append(r)
        Xn, Yn, rn = [], [], []
        vistos = set()
        for r in runs:
            g = grupos.get(CHAVE[int(r)], [r])
            if len(g) > 1 and dob[r] is not None:
                if CHAVE[int(r)] in vistos:
                    continue
                vistos.add(CHAVE[int(r)])
                Xn.append(X[g[0]]); Yn.append(np.nanmean(Y[g], axis=0)); rn.append(g[0])
            else:
                Xn.append(X[r]); Yn.append(Y[r]); rn.append(r)
        X, Y, runs = np.array(Xn), np.array(Yn), np.array(rn)
    Xl, yl, rl, cl = [], [], [], []
    for j in range(Y.shape[1]):
        ok = ~np.isnan(Y[:, j])
        if ok.sum() < 5:
            continue
        Xl.append(X[ok]); yl.append(Y[ok, j]); rl.append(runs[ok]); cl.append(np.full(ok.sum(), j))
    Xl, yl = np.vstack(Xl), np.concatenate(yl)
    rl, cl = np.concatenate(rl), np.concatenate(cl)
    dl = np.array([DOBRA.get(int(r)) for r in rl], dtype=object)
    ok = np.array([d is not None for d in dl]) & ~np.isin(rl, list(excluir))
    Xl, yl, rl, cl, dl = Xl[ok], yl[ok], rl[ok], cl[ok], dl[ok].astype(int)
    yfit = np.log(yl) if log_alvo else yl
    saida = {}
    for k in sorted(set(dl)):
        va = dl == k
        tr = ~va
        if tr.sum() < 30 or not va.any():
            continue
        pat = pd.Series(yfit[tr]).groupby(cl[tr]).mean()
        esc = pd.Series(yfit[tr]).groupby(cl[tr]).std().fillna(1.0).replace(0, 1.0)
        ger = float(yfit[tr].mean())

        def M(m):
            return np.hstack([Xl[m],
                              pd.Series(cl[m]).map(pat).fillna(ger).to_numpy().reshape(-1, 1),
                              pd.Series(cl[m]).map(esc).fillna(1.0).to_numpy().reshape(-1, 1)])

        s = StandardScaler()
        Xt, Xv = s.fit_transform(M(tr)), s.transform(M(va))
        p = RandomForestRegressor(random_state=semente, **PARAMS_RF).fit(Xt, yfit[tr]).predict(Xv)
        if estimador == "rf+gb":
            p = (p + GradientBoostingRegressor(n_estimators=300, random_state=semente).fit(Xt, yfit[tr]).predict(Xv)) / 2
        if log_alvo:
            p = np.exp(p)
        for i, pos in enumerate(np.where(va)[0]):
            saida[(k, alvos[cl[pos]], int(rl[pos]))] = (p[i], yl[pos])
    return saida


def compara(ref, alt):
    """Ganho pareado por leitura, com intervalo bootstrap e Wilcoxon."""
    comum = sorted(set(ref) & set(alt))
    a = np.array([abs(ref[c][0] - ref[c][1]) for c in comum])
    b = np.array([abs(alt[c][0] - alt[c][1]) for c in comum])
    dif = a - b
    bs = [np.random.default_rng(s).choice(dif, len(dif), replace=True).mean() for s in range(2000)]
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return dict(n=len(comum), mae_ref=a.mean(), mae=b.mean(), ganho=dif.mean(), ic=(lo, hi),
                p=stats.wilcoxon(a, b).pvalue if dif.any() else 1.0)


def veredito(r):
    lo, hi = r["ic"]
    return "MELHOROU" if lo > 0 else ("PIOROU" if hi < 0 else "INEFICAZ")


print("maquina de avaliacao definida")

### 3.2. Codificações da coluna de perdas, com indicadores de texto novos

A referência é o modelo entregue. Cada variante muda uma coisa. A expectativa foi registrada antes da execução e consta do relatório: alvo em log neutro; remoção dos ensaios sem rótulo neutra ou leve piora; replicatas em média leve piora; ordem sozinha pior que a família; indicadores meta-experimentais dos cabeçalhos neutros ou piores por tenderem a identificador.

In [ ]:
def rotulos13_de(d13):
    cols = [c for c in d13.columns if c.startswith("losses_")]
    out = []
    for i in range(len(d13)):
        m = [c[len("losses_"):] for c in cols if d13.at[i, c] == 1]
        out.append(m[0] if m else "")
    return pd.Series(out)


texto = [(h + " " + str(r)).lower() for h, r in zip(cabecalhos, reason)]
IND_CABECALHO = np.array([[int("doe run" in t) for t in texto],
                          [int(bool(re.search(r"\.\d\b", h)) or "rerun" in h.lower()) for h in cabecalhos],
                          [int("baseline" in t) for t in texto]]).T.astype(float)
IND_DUCT = np.array([[int("new duct" in t) for t in texto]]).T.astype(float)
SEM_ROTULO_TREINO = [i for i in SEM_ROTULO if i in TREINO]
print("suporte dos indicadores dos cabecalhos (doe, rerun, baseline):", IND_CABECALHO.sum(0).astype(int).tolist(),
      "| new duct:", int(IND_DUCT.sum()))
print("ensaios sem rotulo que estao no treino:", SEM_ROTULO_TREINO)

resultados_32 = {}
for area in AREAS:
    # As colunas comuns e os alvos sao identicos nas duas versoes; a de
    # perfil entra apenas como fonte da codificacao de perfil de perdas.
    dperfil, d13 = dados[area]["perfil"], dados[area]["13cat"]
    rot = rotulos13_de(d13)
    comuns = [c for c in dperfil.columns if c.startswith(PREFIXOS) and not c.startswith(("losses_", "perfil_perdas_"))]
    alv = [c for c in alvos_de(dperfil) if dperfil[c].notna().sum() >= 5]
    base = dperfil[comuns].to_numpy(float)
    Y = dperfil[alv].to_numpy(float)
    A = dperfil[[c for c in dperfil.columns if c.startswith("losses_")]].to_numpy(float)
    F = pd.get_dummies(rot.map(familia)).to_numpy(float)
    O = pd.get_dummies(rot.map(ordem)).to_numpy(float)
    Md = rot.str.contains("modified").astype(float).to_numpy().reshape(-1, 1)
    variantes = {
        "M0 referencia, codificacao A": (np.hstack([base, A]), {}),
        "V1 alvo em log": (np.hstack([base, A]), {"log_alvo": True}),
        "V2 sem os ensaios sem rotulo": (np.hstack([base, A]), {"excluir": SEM_ROTULO_TREINO}),
        "V3 replicatas em media": (np.hstack([base, A]), {"media_replicatas": True}),
        "V4b ordem apenas": (np.hstack([base, O]), {}),
        "V4c familia + ordem": (np.hstack([base, F, O]), {}),
        "V4d familia + ordem + modificado (C)": (np.hstack([base, F, O, Md]), {}),
        "V5 A + indicadores dos cabecalhos": (np.hstack([base, A, IND_CABECALHO]), {}),
        "V6 A + new duct": (np.hstack([base, A, IND_DUCT]), {}),
    }
    for nome, (X, kw) in variantes.items():
        r = avalia(X, Y, alv, **kw)
        resultados_32.setdefault(nome, {}).update({(area,) + k: v for k, v in r.items()})
    print(f"  {area}: concluida", flush=True)

In [ ]:
ref32 = resultados_32["M0 referencia, codificacao A"]
TABELA = []
print(f"{'variante':<40}{'MAE':>8}{'ganho':>9}{'IC 95%':>22}{'p':>9}  veredito")
print(f"{'M0 referencia, codificacao A':<40}{np.mean([abs(p - y) for p, y in ref32.values()]):>8.3f}")
for nome, res in resultados_32.items():
    if nome.startswith("M0"):
        continue
    r = compara(ref32, res)
    TABELA.append((nome, r))
    print(f"{nome:<40}{r['mae']:>8.3f}{r['ganho']:>+9.3f}{f'[{r[chr(105)+chr(99)][0]:+.3f}, {r[chr(105)+chr(99)][1]:+.3f}]':>22}{r['p']:>9.4f}  {veredito(r)}")
print("\n(com 8 comparacoes contra a mesma referencia, o limiar de Bonferroni para p e 0,00625)")

### 3.3. Camada de decisão: margem fixa contra margem calibrada

A margem de segurança não altera o modelo nem o erro; altera o compromisso entre falsos aprovados e falsos reprovados. A margem calibrada é o quantil dos resíduos fora da amostra das demais dobras, por região. A expectativa registrada era de ganho no extremo de segurança e neutralidade no ponto de operação entregue.

In [ ]:
oof = pd.DataFrame([{"area": k[0], "dobra": k[1], "comp": k[2], "real": v[1], "pred": v[0]}
                    for k, v in ref32.items()])
oof["res"] = oof["real"] - oof["pred"]
oof["lim"] = oof["comp"].map(LIM)
c = oof[oof["lim"].notna()].copy()
atinge = (c["real"] >= c["lim"]).to_numpy()
print(f"leituras com limite: {len(c)} | que atingem: {int(atinge.sum())}\n")


def decide(margem):
    sinal = (c["pred"].to_numpy() + margem) >= c["lim"].to_numpy()
    fa, fr = int((atinge & ~sinal).sum()), int((~atinge & sinal).sum())
    return fa, fr, 100 * (atinge.sum() - fa) / atinge.sum()


print("MARGEM FIXA")
print(f"{'margem':>8}{'falsos aprovados':>18}{'falsos reprovados':>19}{'deteccao':>10}")
for m in (5, 8, 10, 12, 15):
    fa, fr, det = decide(m)
    print(f"{m:>8}{fa:>18}{fr:>19}{det:>9.1f}%")

print("\nMARGEM CALIBRADA POR REGIAO (quantil dos residuos das demais dobras)")
print(f"{'cobertura':>10}{'falsos aprovados':>18}{'falsos reprovados':>19}{'deteccao':>10}{'margem media':>14}")
CONFORMAL = {}
for q in (0.85, 0.90, 0.95):
    mg = np.array([np.quantile(oof[(oof["area"] == a) & (oof["dobra"] != k)]["res"], q)
                   for a, k in zip(c["area"], c["dobra"])])
    mg = np.maximum(mg, 0)
    sinal = (c["pred"].to_numpy() + mg) >= c["lim"].to_numpy()
    fa, fr = int((atinge & ~sinal).sum()), int((~atinge & sinal).sum())
    CONFORMAL[q] = (fa, fr)
    print(f"{q:>10.2f}{fa:>18}{fr:>19}{100 * (atinge.sum() - fa) / atinge.sum():>9.1f}%{mg.mean():>14.2f}")
fa15, fr15, _ = decide(15)
print(f"\nno extremo de seguranca, com cerca de {fr15} falsos reprovados: margem fixa deixa {fa15} falsos aprovados,")
print(f"a calibrada a 95% deixa {CONFORMAL[0.95][0]} com {CONFORMAL[0.95][1]} falsos reprovados.")

### 3.4. Reprodução do modelo final, com seleção aninhada

As duas alterações com ganho afirmado, a codificação da coluna de perdas escolhida por região e a média de Random Forest com Gradient Boosting onde vence, são reproduzidas com a escolha refeita dentro de cada dobra, em três sementes.

In [ ]:
formulas = {}
escolhas = defaultdict(list)
for semente in (42, 1, 2):
    erro = {}
    for area in AREAS:
        # As colunas comuns e os alvos sao identicos nas duas versoes; a de
        # perfil entra apenas como fonte da codificacao de perfil de perdas.
        dperfil, d13 = dados[area]["perfil"], dados[area]["13cat"]
        rot = rotulos13_de(d13)
        comuns = [c for c in dperfil.columns if c.startswith(PREFIXOS) and not c.startswith(("losses_", "perfil_perdas_"))]
        alv = [c for c in alvos_de(dperfil) if dperfil[c].notna().sum() >= 5]
        base = dperfil[comuns].to_numpy(float)
        Y = dperfil[alv].to_numpy(float)
        cods = {"A": dperfil[[c for c in dperfil.columns if c.startswith("losses_")]].to_numpy(float),
                "B": d13[[c for c in d13.columns if c.startswith("losses_")]].to_numpy(float),
                "C": np.hstack([pd.get_dummies(rot.map(familia)).to_numpy(float),
                                pd.get_dummies(rot.map(ordem)).to_numpy(float),
                                rot.str.contains("modified").astype(float).to_numpy().reshape(-1, 1)])}
        for cod, ext in cods.items():
            for est in ("rf", "rf+gb"):
                r = avalia(np.hstack([base, ext]), Y, alv, semente=semente, estimador=est)
                erro[(area, f"{cod}|{est}")] = {k: abs(v[0] - v[1]) for k, v in r.items()}
        print(f"  semente {semente} | {area}", flush=True)
    for area in AREAS:
        chaves = sorted(erro[(area, "A|rf")])
        for k in sorted({ch[0] for ch in chaves}):
            cand = {cfg[1]: np.mean([v for ch, v in erro[cfg].items() if ch[0] != k])
                    for cfg in erro if cfg[0] == area}
            melhor = min(cand, key=cand.get)
            escolhas[area].append(melhor)
            for ch in chaves:
                if ch[0] == k:
                    formulas.setdefault(("M0", semente), {})[(area,) + ch] = erro[(area, "A|rf")][ch]
                    formulas.setdefault(("M3", semente), {})[(area,) + ch] = erro[(area, melhor)][ch]

print(f"\n{'semente':>8}{'M0 entregue':>13}{'M3 final':>10}{'ganho':>8}")
ganhos = []
for s in (42, 1, 2):
    m0 = np.mean(list(formulas[("M0", s)].values()))
    m3 = np.mean(list(formulas[("M3", s)].values()))
    ganhos.append(m0 - m3)
    print(f"{s:>8}{m0:>13.3f}{m3:>10.3f}{m0 - m3:>+8.3f}")
print(f"ganho medio {np.mean(ganhos):+.3f} C | positivo em {sum(g > 0 for g in ganhos)} de 3 sementes  [afirmado: +0,32]")
print("\nconfiguracao escolhida por regiao (3 sementes x 17 dobras):")
for area in AREAS:
    cont = Counter(escolhas[area])
    print(f"  {area:<14} " + " | ".join(f"{a}: {b}" for a, b in cont.most_common(2)))

### 3.5. Consolidação

A tabela abaixo é gerada a partir dos resultados deste notebook. Possibilidades que dependem da parceira ou que não são testáveis com os dados atuais constam do relatório, com a pergunta exata que as destravaria.

In [ ]:
print(f"{'possibilidade':<44}{'ganho':>8}{'IC 95%':>20}  veredito")
for nome, r in TABELA:
    print(f"{nome:<44}{r['ganho']:>+8.3f}{f'[{r[chr(105)+chr(99)][0]:+.2f}, {r[chr(105)+chr(99)][1]:+.2f}]':>20}  {veredito(r)}")
print(f"{'M3 codificacao e estimador por regiao':<44}{np.mean(ganhos):>+8.3f}{'3 de 3 sementes':>20}  {'MELHOROU' if all(g > 0 for g in ganhos) else 'INEFICAZ'}")
print(f"{'margem calibrada por regiao, a 95%':<44}{'':>8}{f'{fa15} -> {CONFORMAL[0.95][0]} falsos aprov.':>20}  MELHOROU no extremo de seguranca")

### 3.6. Testes confirmatórios pré-registrados

Duas pistas surgiram de leitura por região depois de vista a tabela agregada, o que as torna exploratórias. Cada uma recebe aqui um teste com hipótese e critério declarados antes da execução, sobre dez sementes. O critério é duplo: intervalo de confiança de 95 por cento acima de zero e ganho positivo em pelo menos oito das dez sementes.

A primeira hipótese é que, na região das bobinas, o alvo em logaritmo reduz o erro em relação à escala original. A segunda é que, sobre o alvo em logaritmo, a média de Random Forest com Gradient Boosting supera o Random Forest sozinho, isto é, que os dois ganhos se somam.

As células são autocontidas e podem ser executadas isoladamente.

In [ ]:

import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
from scipy import stats
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing import StandardScaler

_B = "../../colabs_S4/"
_B_PERFIL = "../"
_part = json.load(open("../preparacao_testes/particao_treino_teste.json", encoding="utf-8"))
_DOBRA = {int(k): v for k, v in _part["dobra_por_run_idx"].items()}
_PREF = ("blower_direction_", "housing_", "board_position_", "run_case_", "losses_",
         "perfil_perdas_", "deflector_hole_", "material_coil_support_", "cfg_",
          "reason_", "loss_profile_order_")


def _longo(area, codificacao):
    # A codificacao A usa as colunas de perda da versao de perfil e a B as da
    # versao de treze categorias; ler as duas do mesmo arquivo tornaria as duas
    # codificacoes identicas e esvaziaria a comparacao.
    dperfil = pd.read_csv(_B_PERFIL + f"dataset_{area}_perfil.csv")
    d13 = pd.read_csv(_B + f"dataset_{area}_13cat.csv")
    dp = d13  # base adotada: colunas comuns e alvos
    comuns = [c for c in dp.columns if c.startswith(_PREF) and not c.startswith(("losses_", "perfil_perdas_"))]
    fonte = dperfil if codificacao == "A" else d13
    ext = fonte[[c for c in fonte.columns if c.startswith("losses_")]]
    X = np.hstack([dp[comuns].to_numpy(float), ext.to_numpy(float)])
    alv = [c for c in dp.columns if not c.startswith(_PREF) and dp[c].notna().sum() >= 5]
    Y = dp[alv].to_numpy(float)
    Xl, yl, rl, cl = [], [], [], []
    for j in range(Y.shape[1]):
        ok = ~np.isnan(Y[:, j])
        Xl.append(X[ok]); yl.append(Y[ok, j]); rl.append(np.arange(len(dp))[ok]); cl.append(np.full(ok.sum(), j))
    Xl, yl, rl, cl = np.vstack(Xl), np.concatenate(yl), np.concatenate(rl), np.concatenate(cl)
    dl = np.array([_DOBRA.get(int(r)) for r in rl], dtype=object)
    ok = np.array([d is not None for d in dl])
    return Xl[ok], yl[ok], cl[ok], dl[ok].astype(int)


def _erros(area, codificacao, sementes, log_alvo, combinar):
    Xl, yl, cl, dl = _longo(area, codificacao)
    yfit = np.log(yl) if log_alvo else yl
    por_semente = []
    for sem in sementes:
        e = []
        for k in sorted(set(dl)):
            va = dl == k
            tr = ~va
            if tr.sum() < 30 or not va.any():
                continue
            pat = pd.Series(yfit[tr]).groupby(cl[tr]).mean()
            esc = pd.Series(yfit[tr]).groupby(cl[tr]).std().fillna(1.0).replace(0, 1.0)
            ger = float(yfit[tr].mean())
            M = lambda m: np.hstack([Xl[m], pd.Series(cl[m]).map(pat).fillna(ger).to_numpy().reshape(-1, 1),
                                     pd.Series(cl[m]).map(esc).fillna(1.0).to_numpy().reshape(-1, 1)])
            s = StandardScaler()
            Xt, Xv = s.fit_transform(M(tr)), s.transform(M(va))
            p = RandomForestRegressor(n_estimators=300, random_state=sem, n_jobs=-1).fit(Xt, yfit[tr]).predict(Xv)
            if combinar:
                p = (p + GradientBoostingRegressor(n_estimators=300, random_state=sem).fit(Xt, yfit[tr]).predict(Xv)) / 2
            if log_alvo:
                p = np.exp(p)
            e.extend(np.abs(p - yl[va]))
        por_semente.append(np.array(e))
    return por_semente


def _confirma(rotulo, ref, alt):
    a, b = np.mean(ref, axis=0), np.mean(alt, axis=0)
    dif = a - b
    bs = [np.random.default_rng(s).choice(dif, len(dif), replace=True).mean() for s in range(4000)]
    lo, hi = np.percentile(bs, [2.5, 97.5])
    por_sem = np.array([ref[i].mean() - alt[i].mean() for i in range(len(ref))])
    print(rotulo)
    print(f"  MAE {a.mean():.3f} -> {b.mean():.3f} | ganho {dif.mean():+.4f} C | IC95 [{lo:+.4f}, {hi:+.4f}] | "
          f"wilcoxon p={stats.wilcoxon(a, b).pvalue:.4f}")
    print(f"  ganho por semente: {np.round(por_sem, 3)} | positivas {int((por_sem > 0).sum())} de {len(por_sem)}")
    print(f"  VEREDITO: {'CONFIRMADO' if lo > 0 and (por_sem > 0).sum() >= 8 else 'NAO CONFIRMADO'}\n")
    return a.mean(), b.mean()


SEM = range(10)
lin_A = _erros("coil_temps", "A", SEM, log_alvo=False, combinar=False)
log_A = _erros("coil_temps", "A", SEM, log_alvo=True, combinar=False)
_confirma("HIPOTESE 1, coil_temps, codificacao A: alvo em log contra escala original", lin_A, log_A)

lin_B = _erros("coil_temps", "B", SEM, log_alvo=False, combinar=False)
log_B = _erros("coil_temps", "B", SEM, log_alvo=True, combinar=False)
log_B_gb = _erros("coil_temps", "B", SEM, log_alvo=True, combinar=True)
_confirma("HIPOTESE 2, coil_temps, codificacao B: log com RF/GB contra log com RF", log_B, log_B_gb)

print("PARA SITUAR, coil_temps, codificacao B, mesmas 10 sementes:")
for rot, e in [("linear + RF (entregue)", lin_B), ("log + RF", log_B), ("log + RF/GB", log_B_gb)]:
    print(f"  {rot:<26} {np.mean(e, axis=0).mean():.3f} C")

print("\nSECUNDARIO, alvo em log nas demais regioes, 3 sementes, sem valor confirmatorio:")
for a in ["ct_knobs", "ipc_comp", "ipc_cooling", "main_vent_sys", "mainback", "oven_hmi"]:
    la = _erros(a, "A", range(3), False, False)
    lg = _erros(a, "A", range(3), True, False)
    print(f"  {a:<14} ganho do log {np.mean(la, axis=0).mean() - np.mean(lg, axis=0).mean():+.3f} C")


O primeiro teste confirma a transformação do alvo na região das bobinas. O segundo não confirma o acúmulo: os dois ganhos capturam a mesma estrutura multiplicativa do erro em temperatura alta, e o logaritmo é a alteração mais simples e de maior efeito. A recomendação para essa região passa a ser alvo em logaritmo com a codificação de treze categorias.